# Technique 4 — Providing Examples

Showing beats telling. Giving Claude sample **input/output pairs** ("one-shot" with one example, "multi-shot" with several) is one of the most effective techniques — it demonstrates the exact format, tone, and edge-case handling you want instead of describing it in words.

**Why examples help:**
- Capture corner cases (e.g. sarcasm in sentiment analysis: *"Yeah, best movie since Plan 9"* is actually negative)
- Pin down complex output formats (specific JSON, specific structure)
- Show the exact style/tone
- Wrap each example in **XML tags** — `<sample_input>` / `<ideal_output>` — so Claude knows what each part is

**Pro tips from the lesson:**
- Mine your **highest-scoring eval outputs** for examples — a response that scored 10 is a ready-made `<ideal_output>`.
- Don't just paste the pair — **explain why the output is good** ("well-structured, detailed quantities, aligns with goals"). That teaches the reasoning, not just the shape.

This brings the meal-plan prompt to its **final** form: clear-and-direct opening + `<athlete_information>` + guidelines + a one-shot example with commentary. Same harness, same `dataset.json`.

> On Haiku 4.5 the score is already near the ceiling, so expect a small move at most — but on a weaker model this is where the course's prompt becomes reliably excellent.

## Load the harness + existing dataset

In [1]:
import sys
import os
from dotenv import find_dotenv

REPO_ROOT = os.path.dirname(find_dotenv())
SECTION = os.path.join(REPO_ROOT, "building-with-the-claude-api", "03-prompt-engineering")
if SECTION not in sys.path:
    sys.path.insert(0, SECTION)

from prompt_evaluator import PromptEvaluator, add_user_message, chat

DATASET = os.path.join(SECTION, "dataset.json")
evaluator = PromptEvaluator(max_concurrent_tasks=3)
print("Harness loaded. Reusing dataset ->", DATASET)

Harness loaded. Reusing dataset -> c:\Development\anthropic-cert\building-with-the-claude-api\03-prompt-engineering\dataset.json


## Final prompt — add a one-shot example

The `<sample_input>` / `<ideal_output>` pair shows Claude a complete, well-formed meal plan, and the trailing sentence explains *why* it's good. Everything from the previous techniques stays.

In [2]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information>
- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned

Here is an example with a sample input and an ideal output:
<sample_input>
height: 170
weight: 70
goal: Maintain fitness and improve cholesterol levels
restrictions: High cholesterol
</sample_input>
<ideal_output>
Here is a one-day meal plan for an athlete aiming to maintain fitness and improve cholesterol levels:

*   **Calorie Target:** Approximately 2500 calories
*   **Macronutrient Breakdown:** Protein (140g), Fat (70g), Carbs (340g)

**Meal Plan:**

*   **Breakfast (7:00 AM):** Oatmeal (80g dry weight) with berries (100g) and walnuts (15g). Skim milk (240g).
    *   Protein: 15g, Fat: 15g, Carbs: 60g
*   **Mid-Morning Snack (10:00 AM):** Apple (150g) with almond butter (30g).
    *   Protein: 7g, Fat: 18g, Carbs: 25g
*   **Lunch (1:00 PM):** Grilled chicken breast (120g) salad with mixed greens (150g), cucumber (50g), tomato (50g), and a light vinaigrette dressing (30g). Whole wheat bread (60g).
    *   Protein: 40g, Fat: 15g, Carbs: 70g
*   **Afternoon Snack (4:00 PM):** Greek yogurt (170g, non-fat) with a banana (120g).
    *   Protein: 20g, Fat: 0g, Carbs: 40g
*   **Dinner (7:00 PM):** Baked salmon (140g) with steamed broccoli (200g) and quinoa (75g dry weight).
    *   Protein: 40g, Fat: 20g, Carbs: 80g
*   **Evening Snack (9:00 PM):** Small handful of almonds (20g).
    *   Protein: 8g, Fat: 12g, Carbs: 15g

This meal plan prioritizes lean protein sources, whole grains, fruits, and vegetables, while limiting saturated and trans fats to support healthy cholesterol levels.
</ideal_output>
This example meal plan is well-structured, provides detailed information on food choices and quantities, and aligns with the athlete's goals and restrictions.
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## Re-evaluate

Same dataset, same criteria. Compare to the XML-tags step — on Haiku expect a small nudge (or noise), since we're already near the top of the scale.

In [3]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file=DATASET,
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file=os.path.join(SECTION, "output-05-providing-examples.json"),
    html_output_file=os.path.join(SECTION, "output-05-providing-examples.html"),
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.333333333333333


## Takeaway — the full technique stack

The meal-plan prompt is now the section's finished article, stacking every technique:

1. **Clear & direct** opening instruction
2. **Specific** output-quality guidelines
3. **XML-tag** structure separating data from instructions
4. **One-shot example** (mined from an ideal output) *with* an explanation of why it's good

Examples are the most powerful lever because they *show* rather than *tell* — subtle requirements that are awkward to describe in instructions get demonstrated directly. Best practices: always wrap examples in XML tags, be explicit that you're showing an example, target your most common failure cases, and explain why the example is ideal.

Next: the **Exercise on prompting**, then the section quiz.